In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/male_players.csv")

/var/folders/v6/c29cbybs71971d_cj04qx27m0000gn/T/ipykernel_9352/3425877027.py:1: DtypeWarning: Columns (108) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/male_players.csv")


In [5]:
df.fifa_version

0         24.0
1         24.0
2         24.0
3         24.0
4         24.0
          ... 
180016    15.0
180017    15.0
180018    15.0
180019    15.0
180020    15.0
Name: fifa_version, Length: 180021, dtype: float64

In [8]:
df.fifa_update.unique()

array([2.])

In [9]:
df = df[df.fifa_version == max(df.fifa_version)]
df.sample()

,player_id,player_url,fifa_version,fifa_update,update_as_of,short_name,long_name,player_positions,overall,potential,...,ldm,cdm,rdm,rwb,lb,lcb,cb,rcb,rb,gk
1789,256196,/player/256196/willian-pacho/240002,24.0,2.0,2023-09-22,W. Pacho,William Joel Pacho Tenorio,CB,74,84,...,70+2,70+2,70+2,68+2,71+2,74+2,74+2,74+2,71+2,18+2


In [10]:
list(df.columns)

['player_id',
 'player_url',
 'fifa_version',
 'fifa_update',
 'update_as_of',
 'short_name',
 'long_name',
 'player_positions',
 'overall',
 'potential',
 'value_eur',
 'wage_eur',
 'age',
 'dob',
 'height_cm',
 'weight_kg',
 'club_team_id',
 'club_name',
 'league_id',
 'league_name',
 'league_level',
 'club_position',
 'club_jersey_number',
 'club_loaned_from',
 'club_joined_date',
 'club_contract_valid_until_year',
 'nationality_id',
 'nationality_name',
 'nation_team_id',
 'nation_position',
 'nation_jersey_number',
 'preferred_foot',
 'weak_foot',
 'skill_moves',
 'international_reputation',
 'work_rate',
 'body_type',
 'real_face',
 'release_clause_eur',
 'player_tags',
 'player_traits',
 'pace',
 'shooting',
 'passing',
 'dribbling',
 'defending',
 'physic',
 'attacking_crossing',
 'attacking_finishing',
 'attacking_heading_accuracy',
 'attacking_short_passing',
 'attacking_volleys',
 'skill_dribbling',
 'skill_curve',
 'skill_fk_accuracy',
 'skill_long_passing',
 'skill_ball_co

In [11]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [12]:
df_sorted = df.sort_values(['player_id', 'fifa_version', 'fifa_update'])
players = df_sorted.drop_duplicates(subset='player_id', keep='last').reset_index(drop=True)

In [13]:
players = players[~players['player_positions'].str.contains('GK', na=False)].copy()
players['primary_position'] = players['player_positions'].str.split(',').str[0].str.strip()

In [14]:
CORE_STATS = ['pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic']
EXTRA_NUM = ['overall', 'age', 'height_cm']
NUM_COLS = CORE_STATS + EXTRA_NUM

In [15]:
players = players.dropna(subset=NUM_COLS + ['primary_position']).reset_index(drop=True)

In [16]:
positions = sorted(players['primary_position'].unique())
pos2idx = {p: i for i, p in enumerate(positions)}
players['pos_idx'] = players['primary_position'].map(pos2idx)
NUM_POSITIONS = len(positions)

In [17]:
stat_min = players[NUM_COLS].min()
stat_max = players[NUM_COLS].max()
players_norm = players.copy()
players_norm[NUM_COLS] = (players[NUM_COLS] - stat_min) / (stat_max - stat_min + 1e-8)

In [18]:
NUM_ATTRS = len(NUM_COLS)
print(f"Players: {len(players)}, Positions: {NUM_POSITIONS}, Attrs: {NUM_ATTRS}")

Players: 16305, Positions: 14, Attrs: 9


In [19]:
class ScoutingDataset(Dataset):
    def __init__(self, players_norm, num_cols, noise_std=0.05, min_care=2, max_care=6):
        self.stats = players_norm[num_cols].values.astype(np.float32)
        self.pos_idx = players_norm['pos_idx'].values.astype(np.int64)
        self.noise_std = noise_std
        self.min_care = min_care
        self.max_care = max_care
        self.num_attrs = len(num_cols)

    def __len__(self):
        return len(self.stats)

    def __getitem__(self, idx):
        player_vec = self.stats[idx]
        pos = self.pos_idx[idx]

        # Synthetic query: noisy version of player's own stats
        noise = np.random.normal(0, self.noise_std, size=self.num_attrs).astype(np.float32)
        query_target = np.clip(player_vec + noise, 0, 1)

        # Random subset of attributes the scout "cares about"
        n_care = np.random.randint(self.min_care, self.max_care + 1)
        care_idx = np.random.choice(self.num_attrs, size=n_care, replace=False)
        weight = np.zeros(self.num_attrs, dtype=np.float32)
        weight[care_idx] = 1.0

        # Position weighting: scout may or may not filter by position
        pos_matters = np.random.rand() < 0.8  # 80% of the time position matters

        return {
            'player_stats': torch.from_numpy(player_vec),
            'player_pos': torch.tensor(pos, dtype=torch.long),
            'query_target': torch.from_numpy(query_target),
            'query_weight': torch.from_numpy(weight),
            'query_pos': torch.tensor(pos, dtype=torch.long),
            'query_pos_mask': torch.tensor(1.0 if pos_matters else 0.0, dtype=torch.float32),
        }


In [20]:
dataset = ScoutingDataset(players_norm, NUM_COLS)

n_val = int(0.1 * len(dataset))
n_train = len(dataset) - n_val
train_ds, val_ds = torch.utils.data.random_split(dataset, [n_train, n_val])

In [21]:
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

In [22]:
EMB_DIM = 64
POS_EMB_DIM = 16

In [23]:
class PlayerTower(nn.Module):
    def __init__(self, num_attrs, num_positions, pos_emb_dim=POS_EMB_DIM, emb_dim=EMB_DIM):
        super().__init__()
        self.pos_emb = nn.Embedding(num_positions, pos_emb_dim)
        self.net = nn.Sequential(
            nn.Linear(num_attrs + pos_emb_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, emb_dim),
        )

    def forward(self, stats, pos_idx):
        pos_e = self.pos_emb(pos_idx)
        x = torch.cat([stats, pos_e], dim=-1)
        out = self.net(x)
        return F.normalize(out, dim=-1)

In [24]:
class QueryTower(nn.Module):
    def __init__(self, num_attrs, num_positions, pos_emb_dim=POS_EMB_DIM, emb_dim=EMB_DIM):
        super().__init__()
        self.pos_emb = nn.Embedding(num_positions, pos_emb_dim)
        self.no_pos_emb = nn.Parameter(torch.zeros(pos_emb_dim))  # "don't care" position
        # input: target values + weights + position embedding
        self.net = nn.Sequential(
            nn.Linear(num_attrs * 2 + pos_emb_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, emb_dim),
        )

    def forward(self, target, weight, pos_idx, pos_mask):
        pos_e = self.pos_emb(pos_idx)
        pos_e = pos_e * pos_mask.unsqueeze(-1) + self.no_pos_emb * (1 - pos_mask.unsqueeze(-1))
        x = torch.cat([target * weight, weight, pos_e], dim=-1)  # zero out uncared-about targets
        out = self.net(x)
        return F.normalize(out, dim=-1)


In [25]:
class TwoTowerModel(nn.Module):
    def __init__(self, num_attrs, num_positions):
        super().__init__()
        self.player_tower = PlayerTower(num_attrs, num_positions)
        self.query_tower = QueryTower(num_attrs, num_positions)

    def forward(self, batch):
        player_emb = self.player_tower(batch['player_stats'], batch['player_pos'])
        query_emb = self.query_tower(batch['query_target'], batch['query_weight'],
                                      batch['query_pos'], batch['query_pos_mask'])
        return player_emb, query_emb


In [27]:
device = torch.device('mps' if torch.mps.is_available() else 'cpu')
device

device(type='mps')

In [28]:
model = TwoTowerModel(NUM_ATTRS, NUM_POSITIONS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [29]:
def info_nce_loss(player_emb, query_emb, temperature=0.07):
    logits = query_emb @ player_emb.T / temperature  # [B, B]
    labels = torch.arange(logits.size(0), device=logits.device)
    loss_q2p = F.cross_entropy(logits, labels)
    loss_p2q = F.cross_entropy(logits.T, labels)
    return (loss_q2p + loss_p2q) / 2

In [30]:
EPOCHS = 20

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    with torch.set_grad_enabled(train):
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            player_emb, query_emb = model(batch)
            loss = info_nce_loss(player_emb, query_emb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * batch['player_stats'].size(0)
    return total_loss / len(loader.dataset)

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f}")

Epoch 01 | train_loss 3.8466 | val_loss 3.5757
Epoch 02 | train_loss 3.6179 | val_loss 3.5651
Epoch 03 | train_loss 3.3671 | val_loss 3.0878
Epoch 04 | train_loss 3.0049 | val_loss 2.8539
Epoch 05 | train_loss 2.8880 | val_loss 2.8029
Epoch 06 | train_loss 2.8017 | val_loss 2.7764
Epoch 07 | train_loss 2.7246 | val_loss 2.7105
Epoch 08 | train_loss 2.6673 | val_loss 2.6190
Epoch 09 | train_loss 2.6071 | val_loss 2.5869
Epoch 10 | train_loss 2.5733 | val_loss 2.4846
Epoch 11 | train_loss 2.5260 | val_loss 2.5243
Epoch 12 | train_loss 2.4788 | val_loss 2.4694
Epoch 13 | train_loss 2.4357 | val_loss 2.3455
Epoch 14 | train_loss 2.3817 | val_loss 2.2878
Epoch 15 | train_loss 2.3571 | val_loss 2.3301
Epoch 16 | train_loss 2.3408 | val_loss 2.2653
Epoch 17 | train_loss 2.3030 | val_loss 2.2308
Epoch 18 | train_loss 2.2678 | val_loss 2.2931
Epoch 19 | train_loss 2.2504 | val_loss 2.1800
Epoch 20 | train_loss 2.2227 | val_loss 2.1839


In [31]:
model.eval()
with torch.no_grad():
    all_stats = torch.from_numpy(dataset.stats).to(device)
    all_pos = torch.from_numpy(dataset.pos_idx).to(device)
    all_player_emb = model.player_tower(all_stats, all_pos)  # [N, emb_dim]

def scout_search(desired: dict, position: str = None, top_k: int = 10):
    """
    desired: dict of {attr_name: target_value_0_100}, e.g. {'defending': 85, 'physic': 80}
    position: optional position string, e.g. 'CB'
    """
    target = np.zeros(NUM_ATTRS, dtype=np.float32)
    weight = np.zeros(NUM_ATTRS, dtype=np.float32)
    for attr, val in desired.items():
        i = NUM_COLS.index(attr)
        norm_val = (val - stat_min[attr]) / (stat_max[attr] - stat_min[attr] + 1e-8)
        target[i] = norm_val
        weight[i] = 1.0

    pos_mask = 1.0 if position else 0.0
    pos_idx = pos2idx.get(position, 0) if position else 0

    with torch.no_grad():
        q_target = torch.tensor(target).unsqueeze(0).to(device)
        q_weight = torch.tensor(weight).unsqueeze(0).to(device)
        q_pos = torch.tensor([pos_idx], dtype=torch.long).to(device)
        q_mask = torch.tensor([pos_mask]).to(device)
        query_emb = model.query_tower(q_target, q_weight, q_pos, q_mask)

        sims = (query_emb @ all_player_emb.T).squeeze(0)  # [N]
        top_idx = torch.topk(sims, top_k).indices.cpu().numpy()

    results = players.iloc[top_idx][['short_name', 'primary_position', 'overall', 'age'] + CORE_STATS].copy()
    results['similarity'] = sims[top_idx].cpu().numpy()
    return results.reset_index(drop=True)

In [33]:
# 1. Classic ball-playing center-back
scout_search({'defending': 85, 'physic': 82, 'passing': 78}, position='CB', top_k=10)

,short_name,primary_position,overall,age,pace,shooting,passing,dribbling,defending,physic,similarity
0,V. Lindelöf,CB,80,28,59.0,51.0,73.0,72.0,81.0,76.0,0.791137
1,A. Laporte,CB,85,29,61.0,50.0,73.0,69.0,86.0,78.0,0.786180
2,A. Bastoni,CB,85,24,73.0,35.0,72.0,73.0,86.0,83.0,0.784220
3,M. Senesi,CB,76,26,59.0,42.0,69.0,72.0,76.0,75.0,0.780610
4,Yeray,CB,81,28,64.0,49.0,68.0,65.0,83.0,76.0,0.779968
5,M. de Ligt,CB,86,23,66.0,61.0,64.0,68.0,85.0,86.0,0.776861
6,Rúben Dias,CB,89,26,62.0,39.0,66.0,69.0,89.0,87.0,0.775421
7,J. Stones,CB,85,29,72.0,52.0,75.0,77.0,85.0,77.0,0.774168
8,M. Ginter,CB,84,29,60.0,59.0,71.0,66.0,85.0,80.0,0.770796
9,A. Christensen,CB,83,27,67.0,32.0,68.0,71.0,85.0,76.0,0.770321


In [34]:
scout_search({'pace': 90, 'dribbling': 88, 'shooting': 80}, position='RW', top_k=10)

,short_name,primary_position,overall,age,pace,shooting,passing,dribbling,defending,physic,similarity
0,O. Dembélé,RW,86,26,93.0,77.0,81.0,87.0,36.0,57.0,0.878216
1,S. Chukwueze,RW,81,24,89.0,75.0,74.0,84.0,36.0,63.0,0.877502
2,Rodrygo,RW,85,22,88.0,81.0,79.0,86.0,31.0,62.0,0.874590
3,Antony,RW,81,23,83.0,76.0,74.0,87.0,46.0,73.0,0.864561
4,T. Kubo,RW,80,22,84.0,75.0,76.0,83.0,40.0,60.0,0.863061
5,H. Lozano,RW,81,27,93.0,76.0,72.0,82.0,41.0,61.0,0.858592
6,N. González,RW,81,25,87.0,76.0,73.0,83.0,45.0,64.0,0.852483
7,J. Ikoné,RW,77,25,89.0,67.0,72.0,83.0,35.0,61.0,0.849518
8,B. Mbeumo,RW,78,23,83.0,77.0,72.0,78.0,45.0,74.0,0.846930
9,B. Saka,RW,86,21,85.0,81.0,79.0,87.0,60.0,70.0,0.840871


In [35]:
# 3. Box-to-box midfielder, no position filter (let the model figure out role from stats alone)
scout_search({'passing': 82, 'physic': 80, 'defending': 70, 'dribbling': 75}, top_k=10)

,short_name,primary_position,overall,age,pace,shooting,passing,dribbling,defending,physic,similarity
0,T. Alexander-Arnold,RB,86,24,76.0,69.0,90.0,79.0,80.0,73.0,0.816818
1,K. Trippier,RB,85,32,68.0,65.0,86.0,79.0,82.0,72.0,0.815297
2,E. Palacios,CM,80,24,64.0,74.0,77.0,79.0,79.0,73.0,0.814452
3,R. James,RB,84,23,80.0,72.0,83.0,82.0,82.0,81.0,0.807676
4,G. Di Lorenzo,RB,85,29,85.0,67.0,75.0,79.0,82.0,82.0,0.806684
5,Sergi Roberto,RB,80,31,74.0,63.0,79.0,77.0,75.0,73.0,0.805727
6,Ricardo Pereira,RB,79,29,72.0,66.0,77.0,77.0,79.0,70.0,0.801753
7,L. Paredes,CM,77,29,62.0,64.0,81.0,79.0,71.0,75.0,0.800052
8,W. Endo,CM,80,30,67.0,68.0,73.0,80.0,79.0,75.0,0.799818
9,K. Walker,RB,84,33,90.0,63.0,77.0,78.0,79.0,81.0,0.796758


In [36]:
torch.save({
    'model_state_dict': model.state_dict(),
    'pos2idx': pos2idx,
    'stat_min': stat_min.to_dict(),
    'stat_max': stat_max.to_dict(),
    'num_cols': NUM_COLS,
    'core_stats': CORE_STATS,
    'num_attrs': NUM_ATTRS,
    'num_positions': NUM_POSITIONS,
}, '../models/two_tower_scout.pt')

In [37]:
with torch.no_grad():
    all_player_emb_np = all_player_emb.cpu().numpy()

emb_cols = [f'emb_{i}' for i in range(EMB_DIM)]
emb_df = pd.DataFrame(all_player_emb_np, columns=emb_cols)

export_df = pd.concat([
    players[['player_id', 'short_name', 'primary_position', 'overall', 'age'] + CORE_STATS].reset_index(drop=True),
    emb_df
], axis=1)

export_df.to_csv('player_embeddings.csv', index=False)
print(f"Saved {len(export_df)} player embeddings to player_embeddings.csv")

Saved 16305 player embeddings to player_embeddings.csv
